In [1]:
import requests
import json
import os
import time
import csv
from datetime import datetime
import isodate

In [2]:
os.chdir("C:/Users/victo/Desktop/proyecto-metrics-youtube/02_proyecto_principal")

with open("keys.txt", "r") as file:
    linea = file.read().strip()
    API_KEY = linea.split("=",1)[1]

In [3]:
def hacer_peticion(url, params, max_retries=3):
    """
    Realiza una petición GET con reintentos en caso de error.
    
    Args:
        url (str): La URL de la API.
        params (dict): Parámetros de la consulta.
        max_retries (int): Número máximo de intentos.
    
    Returns:
        dict or None: El JSON de la respuesta si es exitosa, None si falla tras reintentos.
    """
    for intento in range(1, max_retries + 1):
        try:
            respuesta = requests.get(url, params=params)
            respuesta.raise_for_status()  # Lanza excepción si el código HTTP no es 200
            return respuesta.json()       # Si llegó aquí, todo bien
        except requests.exceptions.RequestException as e:
            print(f"Intento {intento} falló: {e}")
            if intento < max_retries:
                print(f"Esperando 5 segundos antes de reintentar...")
                time.sleep(5)
            else:
                print(f"Fallo después de {max_retries} intentos.")
                return None

In [4]:
HANDLE = ["@Monitorfantasma", "@hugoxchugox", "@ElRobotdePlaton", "@DiloNomas", "@lahistoriade...1209"]

detalles_videos = []
for handle in HANDLE:
    params = {
        "part" : "contentDetails",
        "forHandle": handle,
        "key": API_KEY
    }

    channel_url = "https://www.googleapis.com/youtube/v3/channels"

    respuesta = requests.get(channel_url, params = params)

    if respuesta.status_code != 200:
        print(f"Error HTTP: {respuesta.status_code}")
        exit(1)

    data = respuesta.json()

    if not data.get("items"):
        print("El canal no existe o no tiene datos públicos.")
        exit(1)

    uploads_playlist_id = data["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    #print(uploads_playlist_id)

    video_id_lista = []
    next_page_token = None

    while True:

        faltantes = 200 - len(video_id_lista)

        if faltantes <= 0:
            break

        result_a_llamar = min(faltantes, 50)

        params_videos = {
            "part": "snippet",
            "playlistId": uploads_playlist_id,
            "maxResults": result_a_llamar,
            "key": API_KEY
        }
        if next_page_token:
            params_videos["pageToken"] = next_page_token
        
        lista_videos_url = "https://www.googleapis.com/youtube/v3/playlistItems"

        
        lista_videos_data = hacer_peticion(lista_videos_url, params = params_videos)

        if lista_videos_data is None:
            print("Error crítico: no se pudo obtener la página después de reintentos.")
            break

        try:
            items = lista_videos_data["items"]
        except KeyError:
            print("La respuesta no contiene 'items'. Estructura inesperada:")
            print(lista_videos_data)
            break
        
        for item in items:
            video_id = item["snippet"]["resourceId"]["videoId"]
            video_id_lista.append(video_id)

        next_page_token = lista_videos_data.get("nextPageToken")
        if not next_page_token or len(video_id_lista) == 200:
            break

    #print(f"Total videos: {len(video_id_lista)}")
    #print(video_id_lista)

    timestamp_extraccion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


    for inicio in range(0, len(video_id_lista), 50):
        bloque = video_id_lista[inicio:inicio+50]
        ids_str = ",".join(bloque)

        params_metricas = {
            "part": "contentDetails,snippet,statistics",
            "id": ids_str,
            "key": API_KEY
        }

        metricas_url = "https://www.googleapis.com/youtube/v3/videos"

        lista_metricas_url = hacer_peticion(metricas_url, params=params_metricas)

        if "items" not in lista_metricas_url:
            print("Advertencia: respuesta sin items", lista_metricas_url)
            continue
        
        for item in lista_metricas_url["items"]:
            detalle_por_video = {}
            detalle_por_video["timestamp"] = timestamp_extraccion
            detalle_por_video["channelTitle"] = item["snippet"].get("channelTitle", None)
            detalle_por_video["id"] = item.get("id", None)
            detalle_por_video["title"] = item["snippet"].get("title", None)
            detalle_por_video["viewCount"] = item["statistics"].get("viewCount", 0)
            detalle_por_video["likeCount"] = item["statistics"].get("likeCount", 0)
            detalle_por_video["commentCount"] = item["statistics"].get("commentCount", 0)
            detalle_por_video["duration"] = item["contentDetails"].get("duration", None)

            if detalle_por_video["duration"]:
                try:
                    duracion_segundos = isodate.parse_duration(detalle_por_video["duration"]).total_seconds()
                except:
                    duracion_segundos = 0
            else:
                duracion_segundos = 0

            if duracion_segundos >= 300:
            
                detalles_videos.append(detalle_por_video)

    #print(detalles_videos)
    time.sleep(1)

In [5]:
nombre_archivo = "info_videos_youtube_raw.csv"

campos = ["timestamp", "channelTitle","id","title", "viewCount", "likeCount", "commentCount", "duration"]

with open(nombre_archivo, 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.DictWriter(archivo, fieldnames=campos)
    writer.writeheader()           # escribe la fila de encabezados
    writer.writerows(detalles_videos)   # escribe todas las filas de una vez